In [0]:
from pyspark.sql import functions as F

customers = spark.table("workspace.silver.customers")
orders = spark.table("workspace.silver.orders")

customer_metrics = (
    orders
    .filter(F.col("order_status").isin("COMPLETED", "SHIPPED"))
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("total_amount").alias("lifetime_value"),
        F.avg("total_amount").alias("average_order_value"),
        F.max("order_date").alias("last_order_date")
    )
)

customer_360 = (
    customers
    .join(customer_metrics, "customer_id", "left")
    .fillna({
        "total_orders": 0,
        "lifetime_value": 0,
        "average_order_value": 0
    })
)

(
    customer_360.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.customer_360")
)

display(customer_360)